# 00b · Descarga corregida de RNA-seq para cuatro subtipos intrínsecos

Genera GSE96058 y TCGA-BRCA con un endpoint homogéneo:
**LumA, LumB, HER2-enriched y Basal-like**.

- GSE96058: PAM50, sin las 136 réplicas técnicas y sin Normal-like.
- TCGA-BRCA: PAM50 RNA-seq para las cuatro clases, solo código TCGA `01`
  (Primary Solid Tumor) y una muestra primaria por paciente.
- Las variables ER/PR/HER2 no se usan para sobrescribir PAM50.


In [ ]:
from pathlib import Path
Path('Data').mkdir(parents=True, exist_ok=True)


In [ ]:
# ======================= IMPORTS =======================
import re, io, requests, pandas as pd, numpy as np, GEOparse as gp, tempfile
from pathlib import Path
from typing import Optional, Tuple, Dict, List, Iterable
from collections import defaultdict
import mygene
import os, shutil
import hashlib

# ======================= CONFIG =======================
USER_AGENT = os.environ.get(
    "INTERGATE_NCBI_EMAIL", "alejandro.alvaro@urjc.es"
)  # contacto identificable recomendado por NCBI
_GEO_DL = "https://www.ncbi.nlm.nih.gov/geo/download/"

CACHE_DIR = Path("./cache_geo").resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ======================= HTTP helpers =======================
def _http_get(url: str, params=None, stream=False, timeout=(10, 600)):
    return requests.get(url, params=params, stream=stream, timeout=timeout,
                        headers={"User-Agent": USER_AGENT})

def _safe_name(s: str) -> str:
    # nombre de fichero estable y corto
    h = hashlib.md5(s.encode("utf-8")).hexdigest()[:10]
    base = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)[:80]
    return f"{base}__{h}"

def _cache_path(kind: str, key: str, suffix: str):
    return CACHE_DIR / kind / f"{_safe_name(key)}{suffix}"

def _ensure_dir(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)
    
def _download_bytes(url: str, params=None, timeout=(10, 600), use_cache=True) -> bytes:
    key = url
    if params:
        # orden estable
        key += "?" + "&".join([f"{k}={params[k]}" for k in sorted(params)])

    cache_file = _cache_path("downloads", key, ".bin")
    _ensure_dir(cache_file)

    if use_cache and cache_file.exists() and cache_file.stat().st_size > 0:
        return cache_file.read_bytes()

    with _http_get(url, params=params, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        data = b"".join(r.iter_content(1024 * 1024))

    if use_cache:
        cache_file.write_bytes(data)

    return data


def _is_url(s: str) -> bool:
    return isinstance(s, str) and s.lower().startswith(("http://", "https://", "ftp://"))

def _guess_sep_by_name(name: str) -> str:
    name = name.lower()
    return "," if (name.endswith(".csv") or name.endswith(".csv.gz")) else "\t"

def _find_ensembl_col(df_raw: pd.DataFrame) -> Optional[str]:
    """Devuelve la 1ª columna que parezca mayoritariamente ENSG."""
    for c in df_raw.columns:
        s = df_raw[c].astype(str).head(2000)
        m = s.str.match(r'^ENSG\d+(?:\.\d+)?$')
        if m.mean() > 0.6:
            return c
    return None

def _extract_ensembl_from_index(idx: pd.Index) -> Tuple[pd.Index, float]:
    """Extrae el primer ENSG presente en cada string del índice. Devuelve (nuevo_index, fracción_con_ENSG)."""
    pat = re.compile(r'(ENSG\d+(?:\.\d+)?)')
    vals, hits = [], 0
    for x in map(str, idx):
        m = pat.search(x)
        if m:
            vals.append(m.group(1)); hits += 1
        else:
            vals.append(None)
    frac = hits / max(1, len(idx))
    return pd.Index(vals, name="GeneID"), frac

def _pick_id_col(df: pd.DataFrame) -> str:
    """Detector robusto de columna ID."""
    cols = list(df.columns)
    exact = ["GeneID","gene_id","Gene_Id","geneid","Gene","gene","Symbol","symbol","ID","id",
             "Gene.Name","gene.name","Gene_Symbol","gene_symbol","GeneSymbol","genename","GeneName"]
    for c in exact:
        if c in cols:
            return c
    def norm(s): return re.sub(r"[ _\.\-]+", "", str(s).strip().lower())
    normed = {c: norm(c) for c in cols}
    for c, n in normed.items():
        if n in {"geneid","gene_id"}:
            return c
    for c, n in normed.items():
        if "gene" in n and any(x in n for x in ["id","name","symbol"]):
            return c
    return cols[0]

def enterztpsymbol(df):
    df = df.copy()
    df.index = df.index.astype(str).str.strip()   # asegúrate de que sean strings
    
    mg = mygene.MyGeneInfo()
    # Consulta: tus IDs → símbolo y nombre (humano). Cambia species si no es Homo sapiens.
    hits = mg.querymany(
        df.index.tolist(),
        scopes="entrezgene",
        fields="symbol,name,ensembl.gene",
        species="human",
        as_dataframe=True
    )
    
    # quedarnos con una fila por ID de entrada
    hits = hits[~hits.index.duplicated(keep="first")]
    
    # unir a la expresión
    df_mapped = df.join(hits[["symbol"]], how="left")
    df_mapped.index.name = "entrez_id"
    df = (
        df_mapped
          .dropna(subset=['symbol'])   
          .set_index('symbol')         
    )  
    return df
# ======================= DISCOVERY GEO (type=rnaseq_counts) =======================
def _find_rnaseq_counts_file(gse_id: str, timeout=60) -> Optional[str]:
    """Localiza un .tsv/.txt (.gz) en la página 'type=rnaseq_counts' y prioriza 'raw_counts'."""
    params = {"acc": gse_id, "format": "file", "type": "rnaseq_counts"}
    r = _http_get(_GEO_DL, params=params, timeout=timeout)
    r.raise_for_status()
    html = r.text
    cand = re.findall(r'href="[^"]*?file=([^"&]+?\.(?:tsv|txt)\.gz)"', html, flags=re.I)
    if not cand:
        return None
    raw = [c for c in cand if re.search(r'raw[_-]?counts', c, flags=re.I)]
    return raw[0] if raw else cand[0]

# ======================= Lectores de counts =======================
def _read_counts_from_bytes(data: bytes, filename_hint: str) -> pd.DataFrame:
    compression = "gzip" if filename_hint.lower().endswith(".gz") else None
    sep = _guess_sep_by_name(filename_hint)

    df_raw = pd.read_csv(io.BytesIO(data), sep=sep, header=0,
                         low_memory=False, compression=compression)

    # 1) columna de IDs (robusto)
    id_col = _pick_id_col(df_raw)

    # 2) si NO parece ENSG, intenta encontrar otra columna ENSG
    tmp_idx = df_raw[id_col].astype(str)
    looks_ensg = tmp_idx.str.match(r'^ENSG\d+(?:\.\d+)?$').mean() > 0.6
    if not looks_ensg:
        ensg_col = _find_ensembl_col(df_raw)
        if ensg_col is not None:
            id_col = ensg_col

    # 3) fija índice y convierte numérico (mantén sólo columnas con algo numérico)
    df_raw = df_raw.set_index(id_col)
    num = df_raw.apply(pd.to_numeric, errors="coerce")
    keep = num.columns[~num.isna().all()]
    df = num[keep]

    # 4) limpieza
    df = df.dropna(axis=0, how="all")  # elimina filas 100% NaN (cabeceras tipo 'ENSG', etc.)
    df = df[~df.index.astype(str).str.startswith("__")]  # quita contadores especiales
    return df

def fetch_counts_from_url(url: str, timeout=(10, 600)) -> pd.DataFrame:
    data = _download_bytes(url, params=None, timeout=timeout)
    return _read_counts_from_bytes(data, url)

def fetch_geo_counts_df(gse_id: str, filename: Optional[str] = None) -> pd.DataFrame:
    """Descarga y lee el TSV(.gz) de counts a DataFrame (índice=GeneID) usando la página 'type=rnaseq_counts'."""
    if filename is None:
        filename = _find_rnaseq_counts_file(gse_id)
        if filename is None:
            raise FileNotFoundError(f"No encontré fichero de counts para {gse_id} (type=rnaseq_counts).")
    params = {"format": "file", "type": "rnaseq_counts", "acc": gse_id, "file": filename}
    data = _download_bytes(_GEO_DL, params=params)
    return _read_counts_from_bytes(data, filename)

def fetch_counts_any(gse_id: str, filename: Optional[str]) -> pd.DataFrame:
    """Si filename es URL completa → descarga y parsea esa URL. Si no, usa el flujo GEO estándar."""
    if _is_url(filename):
        return fetch_counts_from_url(filename)
    return fetch_geo_counts_df(gse_id, filename=filename)

# ======================= Detección de tipo de índice =======================
def _looks_like_ensembl(idx: pd.Index) -> bool:
    s = pd.Series(idx.astype(str))
    m = s.str.match(r"^ENSG\d+(?:\.\d+)?$")
    return bool((m.sum() / max(1, len(s))) > 0.6)

def _looks_like_symbol(idx: pd.Index) -> bool:
    s = pd.Series(idx.astype(str))
    m = s.str.match(r"^[A-Za-z][A-Za-z0-9\-\._]+$")
    return bool((m.sum() / max(1, len(s))) > 0.6)

def _annotation_index_is_ensembl(annot: pd.DataFrame) -> bool:
    return _looks_like_ensembl(annot.index)

# ======================= Anotación GEO (Human.GRCh38.p13) =======================
def fetch_geo_annotation_df(filename: str = "Human.GRCh38.p13.annot.tsv.gz") -> pd.DataFrame:
    """Descarga y lee la anotación (índice=GeneID o ENSG, con columna 'symbol') en memoria."""
    params = {"format": "file", "type": "rnaseq_counts", "file": filename}
    data = _download_bytes(_GEO_DL, params=params)
    compression = "gzip" if filename.lower().endswith(".gz") else None
    ann = pd.read_csv(io.BytesIO(data), sep="\t", header=0, low_memory=False,
                      compression=compression, dtype=str)

    # índice (GeneID/ENSG/etc.)
    id_col = next((c for c in ann.columns if c.lower() in {"geneid", "gene_id", "id"}), None)
    if id_col is None:
        # si no hay gene_id/geneid, prueba 'Gene'/'gene' o 'Ensembl'
        for c in ann.columns:
            if str(c).lower() in {"gene", "ensembl", "ensembl_id"}:
                id_col = c
                break
        if id_col is None:
            id_col = ann.columns[0]
    ann = ann.set_index(id_col)

    # columna símbolo
    cand = [c for c in ann.columns if re.search(r"(gene[_ ]?name|gene[_ ]?symbol|^symbol$)", c, flags=re.I)]
    if not cand:
        raise RuntimeError("No encuentro columna de símbolo en la anotación.")
    sym_col = cand[0]
    out = ann[[sym_col]].rename(columns={sym_col: "symbol"})
    out["symbol"] = out["symbol"].astype(str).str.strip()
    return out

# ======================= ENSG → SYMBOL (join o MyGene) =======================
def counts_to_symbol_with_annotation(counts: pd.DataFrame, annot: pd.DataFrame, how="sum") -> pd.DataFrame:
    """Join ENSG→SYMBOL cuando la anotación indexa por ENSG/GeneID compatible."""
    strip = lambda idx: pd.Index([re.sub(r"\.\d+$", "", str(x)) for x in idx], name="GeneID")
    counts2, annot2 = counts.copy(), annot.copy()
    counts2.index, annot2.index = strip(counts2.index), strip(annot2.index)
    merged = counts2.join(annot2[["symbol"]], how="left").dropna(subset=["symbol"])
    if how == "mean":
        agg = merged.groupby("symbol").mean(numeric_only=True)
    elif how == "median":
        agg = merged.groupby("symbol").median(numeric_only=True)
    else:
        agg = merged.groupby("symbol").sum(numeric_only=True)
    agg.index.name = "gene_symbol"
    return agg

def _ensg_to_symbol_mygene(ensg_ids: List[str], chunk_size: int = 300) -> pd.Series:
    """
    Fallback: mapea ENSG → SYMBOL vía MyGene.info usando POST (evita 414) y troceando.
    Devuelve una Series alineada al orden de ensg_ids (sin versión), con NaN si no hay match.
    """
    base = "https://mygene.info/v3/query"

    # limpiar versiones y deduplicar
    base_ids = [re.sub(r"\.\d+$", "", str(x)) for x in ensg_ids]
    uniq = sorted(set(base_ids))
    out_map: Dict[str, str] = {}

    # trocear para no exceder límites
    for i in range(0, len(uniq), chunk_size):
        chunk = uniq[i:i + chunk_size]

        # POST form-encoded evita URLs largas
        payload = {
            "q": ",".join(chunk),
            "scopes": "ensembl.gene",
            "fields": "symbol",
            "species": "human",
            "size": len(chunk),
        }
        r = requests.post(base, data=payload, timeout=60,
                          headers={"User-Agent": USER_AGENT})
        r.raise_for_status()

        data = r.json()
        # mygene puede devolver dict con 'hits' o lista directa
        hits = data.get("hits", data) if isinstance(data, dict) else data

        for hit in hits:
            q = hit.get("query")
            sym = hit.get("symbol")
            if q and sym:
                out_map[q] = sym

    # reconstruir en el mismo orden que ensg_ids (sin versión)
    base_ids_series = pd.Index([re.sub(r"\.\d+$", "", str(x)) for x in ensg_ids], name="GeneID")
    syms = [out_map.get(e, np.nan) for e in base_ids_series]
    return pd.Series(syms, index=base_ids_series, dtype="object")


def counts_to_symbol_if_needed(counts: pd.DataFrame,
                               annot_df: Optional[pd.DataFrame],
                               how: str = "sum",
                               force_map: bool = False) -> pd.DataFrame:
    """
    1) Ya símbolos → colapsa por duplicados.
    2) ENSG + anotación en ENSG → join directo.
    3) ENSG + anotación NO en ENSG → MyGene fallback.
    4) force_map (URL) + índice libre con ENSG embebido → extrae ENSG y aplica 2/3.
    5) Último recurso → colapsa por índice actual.
    """
    # 1) Ya son símbolos
    if _looks_like_symbol(counts.index) and not _looks_like_ensembl(counts.index):
        out = getattr(counts.groupby(counts.index), how)(numeric_only=True)
        out.index.name = "gene_symbol"
        return out

    # 2) Índice ENSG
    if _looks_like_ensembl(counts.index):
        if annot_df is not None and _annotation_index_is_ensembl(annot_df):
            return counts_to_symbol_with_annotation(counts, annot_df, how=how)
        # 3) anotación no ENSG -> MyGene
        sym_series = _ensg_to_symbol_mygene(list(counts.index))
        df = counts.copy()
        df.insert(0, "_symbol", sym_series.values)
        df = df.dropna(subset=["_symbol"])
        if df.empty:
            return df.drop(columns=["_symbol"])
        out = getattr(df.groupby("_symbol"), how)(numeric_only=True)
        out.index.name = "gene_symbol"
        return out

    # 4) force_map: índice libre con ENSG dentro (URLs)
    if annot_df is not None and force_map:
        new_idx, frac = _extract_ensembl_from_index(counts.index)
        if frac > 0.5:
            counts2 = counts.copy()
            counts2.index = new_idx
            counts2 = counts2.loc[counts2.index.notna()]
            if _annotation_index_is_ensembl(annot_df):
                return counts_to_symbol_with_annotation(counts2, annot_df, how=how)
            sym_series = _ensg_to_symbol_mygene(list(counts2.index))
            df = counts2.copy()
            df.insert(0, "_symbol", sym_series.values)
            df = df.dropna(subset=["_symbol"])
            if df.empty:
                return df.drop(columns=["_symbol"])
            out = getattr(df.groupby("_symbol"), how)(numeric_only=True)
            out.index.name = "gene_symbol"
            return out

    # 5) fallback: colapsa por índice tal cual
    out = getattr(counts.groupby(counts.index), how)(numeric_only=True)
    out.index.name = "gene_symbol"
    return out

# ======================= Metadata estilo pData (GEOparse en tmp) =======================
def _first_item(meta: dict, key: str, default=""):
    v = (meta or {}).get(key, default)
    if isinstance(v, (list, tuple)): return v[0] if v else default
    return v

def _norm_key(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[\s/\-]+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s

def expand_characteristics_columns(pheno: pd.DataFrame) -> pd.DataFrame:
    ph = pheno.copy()
    char_cols = [c for c in ph.columns
                 if re.match(r"characteristics(_ch[12])?(\.\d+)?$", str(c), flags=re.I)
                 or str(c).lower().startswith("characteristics_ch")]
    extracted = defaultdict(list); all_keys = set()
    for idx, row in ph.iterrows():
        kv = {}
        for c in char_cols:
            val = row.get(c, None)
            if pd.isna(val) or str(val).strip() == "": continue
            lines = val if isinstance(val, (list, tuple)) else [val]
            flat = []
            for ln in lines:
                if isinstance(ln, str):
                    flat.extend([s for s in re.split(r"\s*;\s*|\s*\|\s*|\n+", ln) if s.strip() != ""])
            for item in flat:
                if ":" in item:
                    k, v = item.split(":", 1)
                    k = _norm_key(k); v = v.strip()
                    kv[k] = (kv.get(k, "") + (" | " if k in kv and kv[k] != v else "") + v).strip(" |")
                else:
                    kv["characteristics_free"] = (kv.get("characteristics_free","") + " | " + str(item).strip()).strip(" |")
        for k, v in kv.items():
            extracted[k].append((idx, v)); all_keys.add(k)
    if not all_keys: return ph
    add = pd.DataFrame(index=ph.index, columns=sorted(all_keys), dtype=object)
    for k, pairs in extracted.items():
        for idx, v in pairs: add.at[idx, k] = v
    for c in list(add.columns):
        if c in ph.columns: add.rename(columns={c: f"{c}_from_char"}, inplace=True)
    return pd.concat([ph, add], axis=1)

def get_gse_metadata(gse) -> pd.DataFrame:
    m = gse.metadata or {}
    row = {
        "gse_accession": gse.name,
        "title":          _first_item(m, "title", ""),
        "summary":        _first_item(m, "summary", ""),
        "overall_design": _first_item(m, "overall_design", ""),
        "type":           _first_item(m, "type", ""),
        "pubmed_id":      _first_item(m, "pubmed_id", ""),
        "submission_date":_first_item(m, "submission_date", ""),
        "status":         _first_item(m, "status", ""),
        "contributor":    ", ".join(m.get("contributor", [])) if "contributor" in m else "",
        "platforms":      ", ".join(sorted(gse.gpls.keys())),
        "n_samples":      len(gse.gsms),
    }
    return pd.DataFrame([row]).set_index("gse_accession")

def make_sample_metadata(gse) -> pd.DataFrame:
    rows = []
    for gsm_id, gsm in gse.gsms.items():
        meta = gsm.metadata or {}
        rows.append({
            "GSM": gsm_id,
            "title": _first_item(meta, "title", ""),
            "organism_ch1": _first_item(meta, "organism_ch1", _first_item(meta, "organism", "")),
            "source_name_ch1": _first_item(meta, "source_name_ch1", _first_item(meta, "source_name", "")),
            "platform_id": _first_item(meta, "platform_id", ""),
            "geo_accession": _first_item(meta, "geo_accession", gsm_id),
            "supplementary_file": "|".join(meta.get("supplementary_file", [])) if "supplementary_file" in meta else "",
        })
    base = pd.DataFrame(rows).set_index("GSM")
    pheno = gse.phenotype_data.copy(); pheno.index.name = "GSM"
    merged = base.join(pheno, how="outer", lsuffix="_gsm", rsuffix="_pheno")

    for col in ["title","organism_ch1","source_name_ch1","platform_id","geo_accession","supplementary_file"]:
        cg, cp = f"{col}_gsm", f"{col}_pheno"
        if cg in merged.columns and cp in merged.columns:
            merged[col] = merged[cg].combine_first(merged[cp]); merged.drop(columns=[cg, cp], inplace=True)
        elif cg in merged.columns:
            merged.rename(columns={cg: col}, inplace=True)
        elif cp in merged.columns:
            merged.rename(columns={cp: col}, inplace=True)

    merged_expanded = expand_characteristics_columns(merged)
    front = [c for c in ["title","platform_id","organism_ch1","source_name_ch1","supplementary_file"]
             if c in merged_expanded.columns]
    resto = [c for c in merged_expanded.columns if c not in front]
    cols = front + resto
    return merged_expanded.loc[:, cols]

def get_geo_metadata_cached(gse_id: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    geo_dir = CACHE_DIR / "geoparse"
    geo_dir.mkdir(parents=True, exist_ok=True)

    # GEOparse reusa si el fichero ya existe en destdir
    gse = gp.get_GEO(geo=gse_id, destdir=str(geo_dir), annotate_gpl=False, silent=True)

    gse_meta_df = get_gse_metadata(gse)
    gsm_meta_df = make_sample_metadata(gse)
    return gse_meta_df, gsm_meta_df

# ======================= Loader de un GSE (todo en memoria) =======================
def load_geo_rnaseq_with_metadata_mem(
    gse_id: str,
    counts_filename: Optional[str] = None,              # puede ser nombre o URL
    annot_filename: Optional[str] = "Human.GRCh38.p13.annot.tsv.gz",
    collapse: str = "sum",
):
    # 1) counts (GEO o URL)
    counts_df = fetch_counts_any(gse_id, counts_filename)

    # 2) metadata
    gse_meta_df, gsm_meta_df = get_geo_metadata_cached(gse_id)

    # 3) anotación y símbolo
    annot_df = None
    if annot_filename:
        try:
            annot_df = fetch_geo_annotation_df(annot_filename)
        except Exception as e:
            print(f"Aviso anotación: {e}")

    # fuerza mapeo si vino por URL (http/ftp)
    force_map = bool(counts_filename and _is_url(counts_filename))

    counts_symbol = counts_to_symbol_if_needed(
        counts_df, annot_df, how=collapse, force_map=force_map
    )

    return {
        "counts": counts_df,
        "counts_symbol": counts_symbol,
        "gse_meta": gse_meta_df,
        "gsm_meta": gsm_meta_df,
        "annot": annot_df,
    }

# ======================= Batch (varios GSE → memoria) =======================
def batch_load_geo_rnaseq_mem(
    jobs: Dict[str, Optional[str]] | Iterable[str],
    annot_filename: Optional[str] = "Human.GRCh38.p13.annot.tsv.gz",
    collapse: str = "sum",
):
    """
    jobs puede ser:
      - lista de GSE: ["GSE_EXAMPLE","GSE_EXAMPLE", ...]  (autodetecta filename)
      - dict {GSE: counts_filename_o_URL_o_None}
    Devuelve: {"results": {GSE: {...}}, "series_meta": DF, "sample_meta": DF}
    """
    if not isinstance(jobs, dict):
        jobs = {gse: None for gse in jobs}

    results = {}
    series_meta_rows, sample_meta_rows = [], []

    for gse_id, fname in jobs.items():
        try:
            res = load_geo_rnaseq_with_metadata_mem(
                gse_id,
                counts_filename=fname,
                annot_filename=annot_filename,
                collapse=collapse,
            )
            results[gse_id] = res
            series_meta_rows.append(res["gse_meta"])
            gm = res["gsm_meta"].copy()
            gm.insert(0, "gse_accession", gse_id)
            sample_meta_rows.append(gm)
            print(f"[OK] {gse_id} ✅")
        except Exception as e:
            print(f"❌[ERROR] {gse_id}: {e}❌")

    series_meta = pd.concat(series_meta_rows) if series_meta_rows else pd.DataFrame()
    sample_meta = pd.concat(sample_meta_rows) if sample_meta_rows else pd.DataFrame()
    return {"results": results, "series_meta": series_meta, "sample_meta": sample_meta}


# ☄️ Descarga RNASEQ

In [ ]:

# ======================= EJEMPLO DE USO (dinámicas como RNASEQ) =======================
rnaseq_jobs = {
    #"GSE164458": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE164nnn/GSE164458/suppl/GSE164458_BrighTNess_RNAseq_log2_Processed_ASTOR.txt.gz",
    "GSE96058": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE96nnn/GSE96058/suppl/GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv.gz",
 
}


batch_rnaseq = batch_load_geo_rnaseq_mem(
    rnaseq_jobs,
    annot_filename="Human.GRCh38.p13.annot.tsv.gz",
    collapse="mean"
)

# variables dinámicas por GSE
for gse_id, res in batch_rnaseq["results"].items():
    globals()[f"counts_{gse_id}"]        = res["counts"]
    globals()[f"counts_symbol_{gse_id}"] = res["counts_symbol"]
    globals()[f"gse_meta_{gse_id}"]      = res["gse_meta"]
    globals()[f"{gse_id}_meta"]          = res["gsm_meta"]
    if res.get("annot") is not None:
        globals()[f"annot_{gse_id}"] = res["annot"]
    print(f"creado counts_{gse_id}, counts_symbol_{gse_id}, gse_meta_{gse_id} y {gse_id}_meta")

## GSE96058 🎉 

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
TARGET_CLASSES = ("LumA", "LumB", "HER2-enriched", "Basal-like")

pam50_candidates = [
    "pam50_subtype",
    "characteristics_ch1.20.pam50 subtype",
]
pam50_col = next((c for c in pam50_candidates if c in GSE96058_meta.columns), None)
if pam50_col is None:
    pam50_col = next(
        (c for c in GSE96058_meta.columns if str(c).lower().endswith("pam50 subtype")),
        None,
    )
if pam50_col is None:
    raise KeyError("GSE96058: no se encontró la columna PAM50 subtype")

title = GSE96058_meta["title"].astype(str).str.strip()
is_technical_replicate = title.str.endswith("repl", na=False)
if int(is_technical_replicate.sum()) != 136:
    raise RuntimeError(
        "GSE96058: se esperaban 136 réplicas técnicas; "
        f"se detectaron {int(is_technical_replicate.sum())}."
    )

GSE96058_meta = GSE96058_meta.loc[~is_technical_replicate].copy()
GSE96058_meta["label_raw"] = GSE96058_meta[pam50_col].astype("string").str.strip()
GSE96058_meta["label"] = GSE96058_meta["label_raw"].map(
    {
        "LumA": "LumA",
        "LumB": "LumB",
        "Her2": "HER2-enriched",
        "Basal": "Basal-like",
    }
)
GSE96058_meta = GSE96058_meta[
    GSE96058_meta["label"].isin(TARGET_CLASSES)
].copy()
GSE96058_meta["batch"] = "GSE96058"
GSE96058_meta["sample"] = GSE96058_meta["title"].astype(str).str.strip()
GSE96058_meta["label_source"] = pam50_col
GSE96058_meta["label_system"] = "PAM50_expression_call"
GSE96058_meta["sample_role"] = "unique_primary_breast_tumor"

if GSE96058_meta["sample"].duplicated().any():
    raise RuntimeError("GSE96058: quedan identificadores biológicos duplicados")
missing = sorted(set(GSE96058_meta["sample"]) - set(counts_symbol_GSE96058.columns))
if missing:
    raise RuntimeError(f"GSE96058: faltan muestras en expresión: {missing[:5]}")
counts_symbol_GSE96058 = counts_symbol_GSE96058.loc[:, GSE96058_meta["sample"]].copy()

observed = (
    GSE96058_meta["label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .astype(int)
    .to_dict()
)
expected = {
    "LumA": 1657,
    "LumB": 729,
    "HER2-enriched": 327,
    "Basal-like": 339,
}
if observed != expected:
    raise RuntimeError(f"GSE96058: observado={observed}, esperado={expected}")
print("GSE96058 auditado:", observed)


In [ ]:
display(
    GSE96058_meta[["sample", "label_raw", "label", "label_system"]].head()
)


In [ ]:
GSE96058_meta = GSE96058_meta[
    [
        "label",
        "batch",
        "sample",
        "label_raw",
        "label_source",
        "label_system",
        "sample_role",
    ]
].copy()


In [ ]:
assert counts_symbol_GSE96058.shape[1] == GSE96058_meta.shape[0] == 3052
assert list(counts_symbol_GSE96058.columns) == GSE96058_meta["sample"].tolist()
print(counts_symbol_GSE96058.shape, GSE96058_meta.shape)


In [ ]:
GSE96058_meta.to_csv("./Data/GSE96058_meta.csv", index=False)
counts_symbol_GSE96058.to_csv("./Data/counts_symbol_GSE96058.csv")
print("GSE96058 guardado sin réplicas y sin Normal-like")


# ☄️ Leemos TCGA_BCRA


In [ ]:
# -*- coding: utf-8 -*-
"""Descarga TCGA-BRCA desde UCSC Xena y crea el endpoint PAM50 intrínseco."""

from __future__ import annotations

import gzip
import io
import json
import re
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import requests

XENA_HOST = "https://tcga.xenahubs.net"
OUTDIR = Path("tcga_brca_auto")
OUTDIR.mkdir(parents=True, exist_ok=True)

TCGA_SAMPLE16_RE = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-\d{2}[A-Z]?)", re.I)
TCGA_PATIENT_RE = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", re.I)

EXPR_DATASET_CANDIDATES = [
    "TCGA.BRCA.sampleMap/HiSeqV2",
    "TCGA.BRCA.sampleMap/HiSeqV2_PANCAN",
]
PHENO_DATASET_CANDIDATES = [
    "TCGA.BRCA.sampleMap/BRCA_clinicalMatrix",
]


def _http_get(url: str, timeout: int = 180) -> bytes:
    response = requests.get(
        url,
        stream=True,
        timeout=timeout,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code} al descargar {url}")
    return response.content


def download_xena_dataset(
    dataset_id_candidates: List[str],
    out_prefix: str,
) -> Tuple[Path, str, str]:
    out_path = OUTDIR / f"{out_prefix}.tsv.gz"
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path, "cached", "cached"

    last_error = None
    for dataset_id in dataset_id_candidates:
        for url in (
            f"{XENA_HOST}/download/{dataset_id}",
            f"{XENA_HOST}/download/{dataset_id}.gz",
        ):
            try:
                raw = _http_get(url)
                if raw[:2] == b"\x1f\x8b":
                    with gzip.GzipFile(fileobj=io.BytesIO(raw)) as gz:
                        text_bytes = gz.read()
                else:
                    text_bytes = raw
                if b"\t" not in text_bytes[:1000]:
                    raise RuntimeError("El contenido no parece TSV")
                with gzip.open(out_path, "wb") as handle:
                    handle.write(text_bytes)
                return out_path, dataset_id, url
            except Exception as exc:
                last_error = exc
    raise RuntimeError(
        f"No se pudo descargar {dataset_id_candidates}. Último error: {last_error}"
    )


def read_tsv_gz(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", low_memory=False)


def _looks_like_tcga_barcodes(values, n: int = 30) -> bool:
    values = [str(value) for value in list(values)[:n]]
    hits = sum(
        bool(TCGA_SAMPLE16_RE.search(value) or TCGA_PATIENT_RE.search(value))
        for value in values
    )
    return hits >= max(1, len(values) // 4)


def load_expression_auto(expr_path: Path) -> pd.DataFrame:
    raw = read_tsv_gz(expr_path)
    if raw.empty:
        raise ValueError("La matriz de expresión está vacía")
    first_col = str(raw.columns[0])

    if _looks_like_tcga_barcodes(raw.columns[1:]):
        expr = raw.set_index(first_col).apply(pd.to_numeric, errors="coerce").T
    elif _looks_like_tcga_barcodes(raw[first_col].astype(str).values):
        expr = raw.set_index(first_col).apply(pd.to_numeric, errors="coerce")
    else:
        raise ValueError("No se pudo inferir la orientación de la expresión")

    expr.index = expr.index.astype(str).str.upper()
    expr.index.name = "sample"
    expr.columns = expr.columns.astype(str)
    return expr


def _find_col(df: pd.DataFrame, patterns: List[str], required: bool = False) -> Optional[str]:
    columns = [str(column) for column in df.columns]
    for pattern in patterns:
        regex = re.compile(pattern, flags=re.I)
        hits = [column for column in columns if regex.search(column)]
        if hits:
            return hits[0]
    if required:
        raise ValueError(f"No se encontró una columna con patrones {patterns}")
    return None


def _extract_sample16(value):
    if pd.isna(value):
        return np.nan
    match = TCGA_SAMPLE16_RE.search(str(value).strip().upper())
    return match.group(1).upper() if match else np.nan


def _extract_patient(value):
    if pd.isna(value):
        return np.nan
    match = TCGA_PATIENT_RE.search(str(value).strip().upper())
    return match.group(1).upper() if match else np.nan


def _norm_pam50(value):
    if pd.isna(value):
        return np.nan
    compact = re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())
    if compact in {"luma", "luminala"}:
        return "LumA"
    if compact in {"lumb", "luminalb"}:
        return "LumB"
    if compact in {"her2", "her2enriched", "her2e"}:
        return "HER2-enriched"
    if compact in {"basal", "basallike"}:
        return "Basal-like"
    return np.nan


def canonicalize_xena_gene_columns(expr: pd.DataFrame):
    """Convierte los identificadores ``SYMBOL|ENTREZ`` de HiSeqV2 a HGNC.

    Xena distribuye muestras por filas después de ``load_expression_auto``.
    Los valores ya están en escala log2, por lo que los símbolos duplicados se
    colapsan por media y nunca por suma.
    """
    raw = pd.Index(expr.columns.astype(str).str.strip(), name="xena_gene_id")
    symbol = raw.str.split("|", n=1).str[0].str.strip()
    valid = (
        symbol.str.match(r"^[A-Za-z][A-Za-z0-9_.-]*$", na=False)
        & ~symbol.str.upper().isin({"NA", "NAN", "NONE"})
        & (symbol != "?")
    )
    out = expr.loc[:, valid].copy()
    out.columns = pd.Index(symbol[valid], name="gene_symbol")
    n_before = int(out.shape[1])
    n_duplicate_rows = int(out.columns.duplicated(keep=False).sum())
    if out.columns.duplicated().any():
        out = out.T.groupby(level=0, sort=True).mean().T
    if out.shape[1] < 1000:
        raise RuntimeError(
            f"TCGA Xena: solo {out.shape[1]} símbolos válidos; revise el formato."
        )
    audit = {
        "raw_gene_identifiers": int(len(raw)),
        "invalid_identifiers_removed": int((~valid).sum()),
        "duplicate_symbol_rows": n_duplicate_rows,
        "unique_gene_symbols": int(out.shape[1]),
        "duplicate_collapse": "mean_on_log2_scale",
    }
    return out, audit


def build_metadata(expr: pd.DataFrame, pheno: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({"sample": expr.index.astype(str)})
    out["sample16"] = out["sample"].map(_extract_sample16)
    out["patient"] = out["sample"].map(_extract_patient)
    out["sample_code"] = out["sample16"].astype("string").str.slice(13, 15)

    ph = pheno.copy()
    ph.columns = [str(column) for column in ph.columns]
    sample_col = _find_col(
        ph,
        [r"^sample(_?id)?$", r"sample", r"barcode", r"submitter"],
        required=True,
    )
    pam_col = _find_col(
        ph,
        [r"^pam50call_rnaseq$", r"pam50call_rnaseq", r"subtype.*pam50", r"pam50"],
        required=True,
    )
    sample_type_col = _find_col(ph, [r"^sample_type$", r"sample type"], required=False)

    ph["sample16"] = ph[sample_col].map(_extract_sample16)
    ph["patient"] = ph[sample_col].map(_extract_patient)
    keep = ["sample16", "patient", pam_col]
    if sample_type_col and sample_type_col not in keep:
        keep.append(sample_type_col)
    phs = ph[keep].copy()
    phs["pam50_raw"] = phs[pam_col]
    phs["pam50"] = phs[pam_col].map(_norm_pam50)
    phs["xena_sample_type"] = (
        phs[sample_type_col].astype("string") if sample_type_col else pd.NA
    )
    phs["_score"] = (
        10 * phs["pam50"].notna().astype(int)
        + phs["xena_sample_type"].notna().astype(int)
    )

    by_sample = (
        phs.dropna(subset=["sample16"])
        .sort_values("_score", ascending=False)
        .drop_duplicates("sample16")
    )
    by_patient = (
        phs.dropna(subset=["patient"])
        .sort_values("_score", ascending=False)
        .drop_duplicates("patient")
    )

    out = out.merge(
        by_sample[["sample16", "pam50_raw", "pam50", "xena_sample_type"]],
        on="sample16",
        how="left",
    )
    out = out.merge(
        by_patient[["patient", "pam50_raw", "pam50", "xena_sample_type"]]
        .add_suffix("_patient"),
        left_on="patient",
        right_on="patient_patient",
        how="left",
    )
    for column in ("pam50_raw", "pam50", "xena_sample_type"):
        out[column] = out[column].where(
            out[column].notna(),
            out[f"{column}_patient"],
        )

    out["is_primary_solid_tumor"] = out["sample_code"].eq("01")
    out["label"] = out["pam50"].where(out["is_primary_solid_tumor"])

    # Si Xena contiene más de un vial primario para el mismo paciente,
    # se conserva uno de forma determinista (01A antes que 01B, etc.).
    out["is_preferred_primary"] = False
    primary_rows = (
        out[out["is_primary_solid_tumor"] & out["patient"].notna()]
        .sort_values(["patient", "sample16", "sample"])
        .drop_duplicates("patient", keep="first")
    )
    out.loc[primary_rows.index, "is_preferred_primary"] = True
    out["use_for_intrinsic4"] = (
        out["is_preferred_primary"] & out["label"].isin(TARGET_CLASSES)
    )
    out["label_raw"] = out["pam50_raw"]
    out["label_source"] = pam_col
    out["label_system"] = "PAM50_RNAseq_expression_call"
    out["sample_role"] = np.where(
        out["is_primary_solid_tumor"],
        "primary_solid_tumor_TCGA_code_01",
        "excluded_nonprimary_sample",
    )

    selected = out[out["use_for_intrinsic4"]]
    if selected["patient"].duplicated().any():
        raise RuntimeError("TCGA: más de una muestra seleccionada por paciente")
    if not selected["sample_code"].eq("01").all():
        raise RuntimeError("TCGA: se seleccionó una muestra no primaria")
    if set(selected["label"].dropna()) != set(TARGET_CLASSES):
        raise RuntimeError(
            f"TCGA: faltan clases; observadas={sorted(selected['label'].dropna().unique())}"
        )

    preferred = [
        "sample",
        "sample16",
        "patient",
        "sample_code",
        "pam50_raw",
        "pam50",
        "label",
        "label_raw",
        "label_source",
        "label_system",
        "sample_role",
        "is_primary_solid_tumor",
        "is_preferred_primary",
        "use_for_intrinsic4",
        "xena_sample_type",
    ]
    return out[preferred]


def main():
    print(">> Descargando expresión TCGA-BRCA")
    expr_path, expr_ds, expr_url = download_xena_dataset(
        EXPR_DATASET_CANDIDATES,
        out_prefix="xena_brca_expression_raw",
    )
    print(">> Descargando fenotipo TCGA-BRCA")
    pheno_path, pheno_ds, pheno_url = download_xena_dataset(
        PHENO_DATASET_CANDIDATES,
        out_prefix="xena_brca_clinical_raw",
    )

    expr = load_expression_auto(expr_path)
    expr = expr.loc[~expr.index.duplicated(keep="first")]
    expr, gene_identifier_audit = canonicalize_xena_gene_columns(expr)
    pheno = read_tsv_gz(pheno_path)
    meta = build_metadata(expr, pheno)

    expr_sxg_path = OUTDIR / "expression_samples_x_genes.tsv.gz"
    expr_gxs_path = OUTDIR / "expression_genes_x_samples.tsv.gz"
    meta_path = OUTDIR / "metadata_tcga_brca_intrinsic4.tsv"
    manifest_path = OUTDIR / "download_manifest_intrinsic4.json"
    expr.to_csv(expr_sxg_path, sep="\t", compression="gzip")
    expr.T.to_csv(expr_gxs_path, sep="\t", compression="gzip")
    meta.to_csv(meta_path, sep="\t", index=False)

    selected = meta[meta["use_for_intrinsic4"]]
    class_counts = {
        str(key): int(value)
        for key, value in selected["label"].value_counts().items()
    }
    manifest = {
        "xena_host": XENA_HOST,
        "expression_dataset_used": str(expr_ds),
        "expression_url_used": str(expr_url),
        "phenotype_dataset_used": str(pheno_ds),
        "phenotype_url_used": str(pheno_url),
        "n_samples_expression": int(expr.shape[0]),
        "n_genes_expression": int(expr.shape[1]),
        "n_selected_unique_primary_tumors": int(len(selected)),
        "class_counts_intrinsic4": class_counts,
        "gene_identifier_contract": gene_identifier_audit,
        "endpoint": {
            "classes": list(TARGET_CLASSES),
            "definition": "PAM50 RNA-seq for all four classes",
            "sample_filter": "TCGA sample code 01; one primary sample per patient",
            "receptor_override": False,
        },
    }
    with open(manifest_path, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)

    print("TCGA seleccionado:")
    display(selected["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0))
    return expr, meta, pheno


expr, meta, pheno = main()


In [ ]:
TCGA_meta = meta[meta["use_for_intrinsic4"]].copy()
assert TCGA_meta["patient"].is_unique
assert TCGA_meta["sample_code"].eq("01").all()
assert set(TCGA_meta["label"]) == set(TARGET_CLASSES)

counts_symbol_TCGA = expr.loc[TCGA_meta["sample"]].T.copy()


In [ ]:
TCGA_meta["batch"] = "TCGA"
TCGA_meta = TCGA_meta[
    [
        "label",
        "batch",
        "sample",
        "patient",
        "sample_code",
        "label_raw",
        "label_source",
        "label_system",
        "sample_role",
    ]
].copy()


In [ ]:
TCGA_meta.index = TCGA_meta["sample"]
display(TCGA_meta.head())
display(TCGA_meta["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0))


In [ ]:
TCGA_meta.to_csv("./Data/TCGA_meta.csv", index=False)
counts_symbol_TCGA.to_csv("./Data/counts_symbol_TCGA.csv")
print("TCGA guardado: solo tumores primarios únicos y PAM50 intrínseco")


In [ ]:
assert set(GSE96058_meta["label"].dropna()) == set(TARGET_CLASSES)
assert set(TCGA_meta["label"].dropna()) == set(TARGET_CLASSES)
assert GSE96058_meta["sample"].is_unique
assert TCGA_meta["patient"].is_unique
assert list(counts_symbol_GSE96058.columns) == GSE96058_meta["sample"].tolist()
assert list(counts_symbol_TCGA.columns) == TCGA_meta["sample"].tolist()
print(
    "RNA-seq definitivo guardado:",
    counts_symbol_GSE96058.shape,
    counts_symbol_TCGA.shape,
)


In [ ]:
rna_audit = pd.concat(
    [
        GSE96058_meta.assign(cohort="GSE96058")[["cohort", "label"]],
        TCGA_meta.assign(cohort="TCGA")[["cohort", "label"]],
    ],
    ignore_index=True,
)
display(pd.crosstab(rna_audit["cohort"], rna_audit["label"]).reindex(columns=TARGET_CLASSES, fill_value=0))
